## 1. Import Libraries & Load Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, classification_report, confusion_matrix, accuracy_score
# Load Data
data_path = "../../data/processed/formatted_data.csv"
try:
    df = pd.read_csv(data_path)
    print(f"Data loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    df = pd.read_csv("D:/AIO/WarmUP/AIO Conquer Warmup/aio-aiconquer-churn-prediction/data/processed/formatted_data.csv")


Data loaded successfully. Shape: (100076, 31)


## 2. Preprocessing

In [2]:
# Drop customer_id
if 'customer_id' in df.columns:
    df = df.drop(columns=['customer_id'])
    print("Dropped 'customer_id'.")

# --- BƯỚC MỚI: FEATURE ENGINEERING ---
# 1. Chi phí trung bình thực tế mỗi tháng (để tìm ra khách hàng bị tính phí ẩn/phạt)
# Cộng 1 để tránh lỗi chia cho 0
df['actual_monthly_cost'] = df['totalcharges'] / (df['tenure'] + 1)

# 2. Mức độ "đắt đỏ" của gói dữ liệu (Tỷ lệ cước phí so với dung lượng dùng)
df['cost_per_gb'] = df['monthlycharges'] / (df['avg_monthly_gb'] + 1)

# 3. Gom nhóm thời gian gắn bó (Tenure Binning) để mô hình dễ nhận dạng khách hàng mới/cũ
bins = [-1, 6, 24, 60, 1000]
labels = ['New', 'Steady', 'Loyal', 'Very_Loyal']
df['tenure_group'] = pd.cut(df['tenure'], bins=bins, labels=labels)

# Categorical columns specified for One-Hot Encoding
cat_cols = ['gender', 'education', 'marital_status', 'contract', 'payment_method', 'tenure_group']
# -------------------------------------

# One-Hot Encoding
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Convert boolean to int for XGBoost
for col in df.columns:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype(int)
        
# Check NaNs
total_nans = df.isnull().sum().sum()
if total_nans > 0:
    print(f"Warning: {total_nans} NaN values found. Filling with median.")
    df = df.fillna(df.median())
else:
    print("Confirmed: No NaN values.")
    
# Split X and y
X = df.drop(columns=['churn'])
y = df['churn']

Dropped 'customer_id'.
Confirmed: No NaN values.


## 3. Train/Test Split (Stratified)

In [3]:
# 80/20 split, stratify=y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train shapes: X={X_train.shape}, y={y_train.shape}")
print(f"Test shapes: X={X_test.shape}, y={y_test.shape}")
print(f"Churn ratio in Train: {y_train.mean():.4f} | Test: {y_test.mean():.4f}")

Train shapes: X=(80060, 43), y=(80060,)
Test shapes: X=(20016, 43), y=(20016,)
Churn ratio in Train: 0.0999 | Test: 0.0999


## 4. Model Configuration & Training

In [4]:
# Using class_weight for imbalanced data in Decision Tree
# class_weight='balanced' automatically adjusts weights inversely proportional to class frequencies
# Or you can manually set it, e.g., class_weight={0: 1, 1: 4.0}

params = {
    'max_depth': 10,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'class_weight': {0: 1, 1: 4.0}, # Similar to scale_pos_weight=4.0
    'random_state': 42
}
print(f"Hyperparameters: {params}")

# Create Decision Tree model
model = DecisionTreeClassifier(**params)

# Train 
print("Training Decision Tree model...")
model.fit(X_train, y_train)
print("Training completed.")


Hyperparameters: {'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2, 'class_weight': {0: 1, 1: 4.0}, 'random_state': 42}
Training Decision Tree model...
Training completed.


## 5. Evaluation & Threshold Tuning

In [5]:
y_pred_prob = model.predict_proba(X_test)[:, 1]

# Default threshold 0.5
print("--- Evaluation at default threshold (0.5) ---")
y_pred_05 = (y_pred_prob >= 0.5).astype(int)
print(classification_report(y_test, y_pred_05))

# Threshold Tuning
thresholds = np.arange(0.1, 0.9, 0.1)
best_thresh = 0.5
best_f1 = 0

print("\n--- Threshold Tuning Grid ---")
for t in thresholds:
    y_pred_t = (y_pred_prob >= t).astype(int)
    rec = recall_score(y_test, y_pred_t)
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_test, y_pred_t)
    print(f"Threshold: {t:.1f} | Recall: {rec:.4f} | Precision: {prec:.4f} | F1: {f1:.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        
print(f"\n=> Selected Best Threshold for F1-Score: {best_thresh:.1f}")

--- Evaluation at default threshold (0.5) ---
              precision    recall  f1-score   support

           0       0.91      0.93      0.92     18017
           1       0.20      0.15      0.17      1999

    accuracy                           0.85     20016
   macro avg       0.55      0.54      0.55     20016
weighted avg       0.84      0.85      0.84     20016


--- Threshold Tuning Grid ---
Threshold: 0.1 | Recall: 0.8999 | Precision: 0.1018 | F1: 0.1829
Threshold: 0.2 | Recall: 0.7344 | Precision: 0.1235 | F1: 0.2115
Threshold: 0.3 | Recall: 0.5718 | Precision: 0.1398 | F1: 0.2247
Threshold: 0.4 | Recall: 0.3492 | Precision: 0.1646 | F1: 0.2237
Threshold: 0.5 | Recall: 0.1536 | Precision: 0.1960 | F1: 0.1722
Threshold: 0.6 | Recall: 0.0705 | Precision: 0.1824 | F1: 0.1017
Threshold: 0.7 | Recall: 0.0455 | Precision: 0.1944 | F1: 0.0738
Threshold: 0.8 | Recall: 0.0190 | Precision: 0.1810 | F1: 0.0344

=> Selected Best Threshold for F1-Score: 0.3


## 6. Final Evaluation

In [6]:
print(f"--- Final Evaluation metrics (Threshold {best_thresh:.1f}) ---")
y_pred_final = (y_pred_prob >= best_thresh).astype(int)

print(f"Recall:    {recall_score(y_test, y_pred_final):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_final):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_final):.4f}")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_final):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_prob):.4f}")

print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred_final)}")

--- Final Evaluation metrics (Threshold 0.3) ---
Recall:    0.5718
Precision: 0.1398
F1-Score:  0.2247
Accuracy:  0.6059
ROC-AUC:   0.6093

Confusion Matrix:
[[10984  7033]
 [  856  1143]]
